In [8]:
import numpy as np
from scapy.all import rdpcap
from collections import defaultdict
from scapy.all import IP, TCP, UDP
from scapy.all import Raw

def group_packets_by_session(packet_list):
    sessions = defaultdict(list)
    
    for pkt in packet_list:
        if IP in pkt:
            src_ip = pkt[IP].src
            dst_ip = pkt[IP].dst
            proto = pkt[IP].proto  # 6 for TCP, 17 for UDP
            
            if TCP in pkt:
                src_port = pkt[TCP].sport
                dst_port = pkt[TCP].dport
            elif UDP in pkt:
                src_port = pkt[UDP].sport
                dst_port = pkt[UDP].dport
            else:
                # Skip ICMP or other protocols without ports
                continue
            
            if src_ip < dst_ip:
                session_key = (src_ip, src_port, dst_ip, dst_port, proto)
            else:
                session_key = (dst_ip, dst_port, src_ip, src_port, proto)
            
            sessions[session_key].append(pkt)

    for session_key in sessions:
        sessions[session_key].sort(key=lambda x: x.time)
            
    return sessions

In [ ]:
def create_image_from_packet(packet, m):

    if not Raw in packet: return np.zeros((int(m ** 0.5),int(m ** 0.5)), dtype=np.uint8)
    
    payload = packet[Raw].load
    payload_m = payload[0:m] + bytes([0 for i in range(max(0,m - len(payload)))])
    img = np.frombuffer(payload_m, dtype=np.uint8).reshape(int(m ** 0.5),int(m ** 0.5)) 
    return img
    
def create_image_from_session(session, n, m):
    assert int(n ** 0.5) * int(n ** 0.5) == n, "n is not a perfect square"
    assert int(m ** 0.5) * int(m ** 0.5) == m, "m is not a perfect square"

    rn = int(n ** 0.5)
    rm = int(m ** 0.5)

    pkt_images = [create_image_from_packet(packet, m) for packet in session[0:n]] 
    padding_images = [np.zeros((rm,rm), dtype=np.uint8) for extra in range(max(0,n - len(session)))]
    
    image = np.stack(pkt_images + padding_images).reshape(rn,rn,rm,rm).transpose(0,2,1,3)
    return image.reshape(rn * rm ,rn * rm)

In [10]:
from torch.utils.data import Dataset

class SessionImageDataset(Dataset):
    def __init__(self, path_to_pcaps: list, labels_of_pcap: list, n , m):
        super().__init__()
        assert len(path_to_pcaps) == len(labels_of_pcap) ,"length of labels and pcap paths should be same"

        self.data = []
        self.labels = []

        for idx in range(len(path_to_pcaps)):
            packets = rdpcap(path_to_pcaps[idx])
            sessions = group_packets_by_session(packets)

            for session in list(sessions.values()):
                self.data.append(create_image_from_session(session , n, m))
                self.labels.append(labels_of_pcap[idx])

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        return self.data[index],self.labels[index]

In [11]:
data = SessionImageDataset(["archive\\Benign\\Facetime.pcap"],[0], 16, 256)